In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import gc
import time
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
DRIVE_BASE = "/content/drive/MyDrive/WiFi Data"

SESSION_PATHS = {
    "DS2": os.path.join(DRIVE_BASE, "Second Data Set"),
    "DS3": os.path.join(DRIVE_BASE, "Third Data Set"),
    "DS5": os.path.join(DRIVE_BASE, "Fifth Data Set"),
    "DS6": os.path.join(DRIVE_BASE, "Sixth Data Set"),
}

for name, path in SESSION_PATHS.items():
    print(f"  {name}: {f'ound' if os.path.isdir(path) else 'NOT FOUND'}")

CSI_COL = 25
TS_COL = 27
NUM_SUBCARRIERS = 64
ROLLING_WINDOW = 10
ROLLING_PASSES = 3
WINDOW_SIZE = 50
STRIDE = 3
PCA_COMPONENTS = 20
ACTIVITIES = ["Walking", "Jumping_Jacks", "Toe_Tap"]
CALIBRATION_RATIO = 0.20
RANDOM_SEED = 42

MAX_EPOCHS = 100
PATIENCE = 7
BATCH_SIZE = 64
AUG_MULTIPLIER = 2
FT_EPOCHS = 20

NULL_SUBCARRIER_INDICES = [0, 1, 2, 3, 4, 5, 32, 59, 60, 61, 62, 63]
ACTIVE_MASK = [i for i in range(NUM_SUBCARRIERS) if i not in NULL_SUBCARRIER_INDICES]
NUM_ACTIVE = len(ACTIVE_MASK)

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(f"Active subcarriers: {NUM_ACTIVE}")

In [ ]:
def parse_csi_string(csi_str):
    try:
        vals = list(map(int, str(csi_str).strip("[] ").replace(",", " ").split()))
        if len(vals) != 128:
            return None
        amplitudes = []
        for i in range(0, 128, 2):
            imag, real = vals[i], vals[i + 1]
            amplitudes.append(np.sqrt(real**2 + imag**2))
        return np.array(amplitudes)
    except:
        return None


def load_session(session_name, session_path):
    csi_file = os.path.join(session_path, "ttyUSB0.csv")
    ann_file = os.path.join(session_path, "annotations.csv")

    csi_df = pd.read_csv(csi_file, header=None, on_bad_lines='skip')

    csi_data, timestamps = [], []
    for idx, row in csi_df.iterrows():
        amp = parse_csi_string(row.iloc[CSI_COL])
        if amp is not None:
            csi_data.append(amp)
            timestamps.append(float(row.iloc[TS_COL]))

    csi_array = np.array(csi_data)[:, ACTIVE_MASK]
    timestamps = np.array(timestamps)

    ann_df = pd.read_csv(ann_file)
    ann_df.columns = [c.strip() for c in ann_df.columns]
    ann_timestamps = ann_df['timestamp'].astype(float).values
    ann_actions = ann_df['current_action'].astype(str).str.strip().values

    sort_idx = np.argsort(ann_timestamps)
    ann_timestamps = ann_timestamps[sort_idx]
    ann_actions = ann_actions[sort_idx]

    segment_idx = np.searchsorted(ann_timestamps, timestamps, side='right') - 1
    segment_idx = np.clip(segment_idx, 0, len(ann_actions) - 1)
    labels = ann_actions[segment_idx]

    print(f"  {session_name}: {csi_array.shape[0]} CSI rows")
    return csi_array, timestamps, labels

In [ ]:
def rolling_mean_denoise(data, window=ROLLING_WINDOW, passes=ROLLING_PASSES):
    df = pd.DataFrame(data)
    for _ in range(passes):
        df = df.rolling(window=window, min_periods=1, center=True).mean()
    return df.values


def per_cycle_local_baseline_subtract(csi_array, labels):
    """
    Local none-baseline subtraction. Each activity rep is preceded by a none
    segment in the collection protocol:
        Walking -> none -> Toe_Tap -> none -> Jumping_Jacks -> none  (x10)

    For each (none, activity) pair, compute the per-subcarrier mean of the none
    segment and subtract it from the following activity segment only. This gives
    each activity rep a baseline calibrated to the background state at that
    specific point in the session.
    """
    n = len(labels)
    out = csi_array.copy()
    kept_mask = np.zeros(n, dtype=bool)

    # find contiguous label segments
    segments = []
    seg_start = 0
    for i in range(1, n + 1):
        if i == n or labels[i] != labels[seg_start]:
            segments.append((seg_start, i, labels[seg_start]))
            seg_start = i

    none_labels = {"none", "None"}
    n_cycles = 0

    for seg_idx in range(len(segments) - 1):
        start_i, end_i, lab_i = segments[seg_idx]
        start_j, end_j, lab_j = segments[seg_idx + 1]

        if lab_i in none_labels and lab_j in ACTIVITIES:
            none_rows = csi_array[start_i:end_i]
            if len(none_rows) == 0:
                continue
            local_baseline = none_rows.mean(axis=0)
            out[start_j:end_j] = csi_array[start_j:end_j] - local_baseline
            kept_mask[start_j:end_j] = True
            n_cycles += 1

    print(f"    {n_cycles} activity cycles with local baseline applied ({kept_mask.sum()} rows)")

    activity_mask = np.isin(labels, ACTIVITIES) & kept_mask
    return out[activity_mask], labels[activity_mask]


def per_session_zscore(csi_array):
    mean = csi_array.mean(axis=0)
    std = csi_array.std(axis=0) + 1e-8
    return (csi_array - mean) / std


def preprocess_session(csi_array, labels):
    csi_denoised = rolling_mean_denoise(csi_array)
    csi_sub, labels_sub = per_cycle_local_baseline_subtract(csi_denoised, labels)
    csi_normed = per_session_zscore(csi_sub)
    return csi_normed, labels_sub


def create_windows(csi_array, labels, window_size=WINDOW_SIZE, stride=STRIDE):
    X_windows, y_windows, start_indices = [], [], []
    for start in range(0, len(csi_array) - window_size + 1, stride):
        window = csi_array[start:start + window_size]
        window_labels = labels[start:start + window_size]
        unique, counts = np.unique(window_labels, return_counts=True)
        mode_label = unique[np.argmax(counts)]
        if counts.max() / counts.sum() >= 0.5:
            X_windows.append(window)
            y_windows.append(mode_label)
            start_indices.append(start)
    return np.array(X_windows), np.array(y_windows), np.array(start_indices)

In [ ]:
def split_calibration_eval(X, y, start_indices, total_rows,
                           calib_ratio=CALIBRATION_RATIO, window_size=WINDOW_SIZE):
    boundary = int(total_rows * calib_ratio)

    calib_mask = (start_indices + window_size) <= boundary
    eval_mask = start_indices >= boundary

    X_calib, y_calib = X[calib_mask], y[calib_mask]
    X_eval, y_eval = X[eval_mask], y[eval_mask]

    dropped = len(X) - calib_mask.sum() - eval_mask.sum()
    return X_calib, y_calib, X_eval, y_eval, dropped

In [ ]:
def augment_data(X, y, multiplier=AUG_MULTIPLIER):
    X_aug_list = [X]
    y_aug_list = [y]
    n = len(X)

    for _ in range(multiplier):
        batch = X.copy()
        choices = np.random.randint(3, size=n)

        noise_mask = choices == 0
        if noise_mask.any():
            stds = batch[noise_mask].std(axis=(1, 2), keepdims=True)
            noise = np.random.normal(0, 0.05 * stds, batch[noise_mask].shape)
            batch[noise_mask] = batch[noise_mask] + noise

        shift_mask = choices == 1
        if shift_mask.any():
            shift_idx = np.where(shift_mask)[0]
            shifts = np.random.randint(-5, 6, size=len(shift_idx))
            for s in np.unique(shifts):
                rows = shift_idx[shifts == s]
                batch[rows] = np.roll(batch[rows], s, axis=1)

        scale_mask = choices == 2
        if scale_mask.any():
            scales = np.random.uniform(0.8, 1.2, size=(scale_mask.sum(), 1, 1))
            batch[scale_mask] = batch[scale_mask] * scales

        X_aug_list.append(batch)
        y_aug_list.append(y.copy())

    return np.concatenate(X_aug_list), np.concatenate(y_aug_list)

In [ ]:
def build_1d_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Conv1D(64, 5, activation='relu', padding='same',
                      kernel_regularizer=regularizers.l2(1e-3),
                      input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.3),

        layers.Conv1D(128, 3, activation='relu', padding='same',
                      kernel_regularizer=regularizers.l2(1e-3)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.4),

        layers.Conv1D(64, 3, activation='relu', padding='same',
                      kernel_regularizer=regularizers.l2(1e-3)),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(0.5),

        layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-3)),
        layers.BatchNormalization(),
        layers.Dropout(0.5),

        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),

        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [ ]:
def run_loo(session_data):
    session_names = list(session_data.keys())
    le = LabelEncoder()
    le.fit(ACTIVITIES)
    num_classes = len(ACTIVITIES)
    results = []
    t_start = time.time()

    for fold_idx, test_name in enumerate(session_names):
        train_names = [s for s in session_names if s != test_name]
        fold_start = time.time()
        elapsed = (fold_start - t_start) / 60
        print(f"\nFold {fold_idx+1}/5: train={train_names}, test={test_name} [{elapsed:.1f} min elapsed]")

        # combine training data and fit PCA
        train_csi = np.concatenate([session_data[s][0] for s in train_names])
        train_labels = np.concatenate([session_data[s][1] for s in train_names])

        pca = PCA(n_components=PCA_COMPONENTS)
        train_pca = pca.fit_transform(train_csi)

        # window and augment training data
        X_train, y_train_str, _ = create_windows(train_pca, train_labels)
        y_train = le.transform(y_train_str)
        y_train_cat = to_categorical(y_train, num_classes)
        X_train_aug, y_train_aug_cat = augment_data(X_train, y_train_cat)
        print(f"  training windows: {X_train.shape[0]} (augmented: {X_train_aug.shape[0]})")

        # train base model
        t0 = time.time()
        model = build_1d_cnn((WINDOW_SIZE, PCA_COMPONENTS), num_classes)
        history = model.fit(
            X_train_aug, y_train_aug_cat,
            epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, validation_split=0.15,
            callbacks=[
                EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
                ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6)
            ],
            verbose=2
        )
        epochs_run = len(history.history['loss'])
        print(f"  base model: {epochs_run} epochs in {time.time()-t0:.0f}s")

        # process test session
        test_csi, test_labels = session_data[test_name]
        test_pca = pca.transform(test_csi)
        X_test, y_test_str, start_indices = create_windows(test_pca, test_labels)
        y_test = le.transform(y_test_str)
        y_test_cat = to_categorical(y_test, num_classes)
        total_rows = len(test_pca)

        # leakage-safe calibration/eval split
        X_calib, y_calib_cat, X_eval, y_eval_cat, n_dropped = split_calibration_eval(
            X_test, y_test_cat, start_indices, total_rows
        )
        y_eval = np.argmax(y_eval_cat, axis=1)

        print(f"  calibration: {len(X_calib)} windows, eval: {len(X_eval)} windows, "
              f"dropped at boundary: {n_dropped}")

        # augment calibration data and fine-tune
        X_calib_aug, y_calib_aug_cat = augment_data(X_calib, y_calib_cat, multiplier=AUG_MULTIPLIER + 2)

        ft_model = tf.keras.models.clone_model(model)
        ft_model.set_weights(model.get_weights())
        ft_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
            loss='categorical_crossentropy', metrics=['accuracy']
        )
        ft_model.fit(X_calib_aug, y_calib_aug_cat, epochs=FT_EPOCHS, batch_size=16, verbose=0)

        # evaluate
        preds = np.argmax(ft_model.predict(X_eval, verbose=0), axis=1)
        acc = accuracy_score(y_eval, preds)

        report = classification_report(y_eval, preds, target_names=le.classes_,
                                       output_dict=True, zero_division=0)
        f1s = ", ".join([f"{a}: {report[a]['f1-score']:.3f}" for a in le.classes_])
        print(f"  {test_name} accuracy: {acc*100:.1f}%")
        print(f"  per-class F1 — {f1s}")

        results.append({
            'fold': fold_idx+1, 'train': train_names, 'test': test_name,
            'accuracy': acc, 'report': report
        })

        del ft_model, model
        tf.keras.backend.clear_session()
        gc.collect()

        fold_time = (time.time() - fold_start) / 60
        avg_fold = (time.time() - t_start) / 60 / (fold_idx + 1)
        remaining = len(session_names) - (fold_idx + 1)
        print(f"  fold time: {fold_time:.1f} min | ETA for remaining {remaining}: ~{avg_fold * remaining:.1f} min")

    total = (time.time() - t_start) / 60
    print(f"\nTotal time: {total:.1f} minutes")
    return results

In [ ]:
def print_results(results):
    print(f"\n{'='*60}")
    print(f"  LOO Cross-Session Results (4-train / 1-test)")
    print(f"{'='*60}")
    print(f"{'Test':<6} {'Train':<22} {'Accuracy':>10}")
    print("-" * 60)

    accs = []
    for r in results:
        t = ",".join(r['train'])
        print(f"{r['test']:<6} {t:<22} {r['accuracy']*100:>9.1f}%")
        accs.append(r['accuracy'])

    print("-" * 60)
    print(f"{'Mean':<6} {'':<22} {np.mean(accs)*100:>9.1f}%")
    print(f"{'Std':<6} {'':<22} {np.std(accs)*100:>9.1f}%")
    print(f"{'='*60}")

In [ ]:
session_data = {}

for name, path in SESSION_PATHS.items():
    csi_array, timestamps, labels = load_session(name, path)
    csi_proc, labels_proc = preprocess_session(csi_array, labels)
    session_data[name] = (csi_proc, labels_proc)
    print(f"  {name}: {csi_proc.shape[0]} activity rows, classes: {np.unique(labels_proc)}")
    print()

print(f"Sessions loaded: {list(session_data.keys())}")

In [ ]:
results = run_loo(session_data)
print_results(results)